  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [15 lines of output]
  The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
  rather than 'sklearn' for pip commands.
  
  Here is how to fix this error in the main use cases:
  - use 'pip install scikit-learn' rather than 'pip install sklearn'
  - replace 'sklearn' by 'scikit-learn' in your pip requirements files
    (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
  - if the 'sklearn' package is used by one of your dependencies,
    it would be great if you take some time to track which package uses
    'sklearn' instead of 'scikit-learn' and report it to their issue tracker
  - as a last resort, set the environment variable
    SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
  
  More information is available at
  https://github.com/scikit-learn/sklearn-pypi-package
  [end of output]
  
  note: This error originates f

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, RandomizedSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score, RocCurveDisplay
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from scipy import stats
#import functions from file




df_soap = pd.read_csv("df_soap.csv")
# print(df_soap.head(1))
X = np.vstack(df_soap["soap"])

# Prepare y array from the bandgaps values
y = np.hstack(df_soap["band_gap"])

print(X.shape, y.shape)
print(X)

In [ ]:
# Start with optimizing the hyperparameters of the Random Forest Classifier
# %%time
# Parameter grid for hyperparameters
p_grid = {"n_estimators": [10, 25, 50], "max_depth": [None, 10, 20, 30]}

# Initializing the Random Forest classifier
rf = RandomForestClassifier(random_state=2)

# Create inner and outer CV fold
inner_cv = KFold(n_splits=5, shuffle=True, random_state=2)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=2)

# Nested CV
gs = GridSearchCV(estimator=rf, param_grid=p_grid, cv=inner_cv)
nested_score = cross_val_score(gs, X=X, y=y, cv=outer_cv)

# Print accuracy and standard error of the nested CV
print(f"Average algorithm accuracy: {np.mean(nested_score):.4f} \n"
      f"and std err: {stats.sem(nested_score):.4f}\n")

# Select hyperparameters with a new 5-fold CV
gs.fit(X,y)
print(f"Selected hyperparameters: {gs.best_params_}\n")

In [ ]:
#Support Vector Regression (SVR)
from sklearn.model_selection import train_test_split
%matplotlib inline
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
X_train3, X_test3, y_train3, y_test3 = train_test_split( X, y, test_size=0.1, random_state=10)
steps = [('scaler', StandardScaler()), ('SVM', SVR())]
pipeline = Pipeline(steps)
grid = GridSearchCV(pipeline, param_grid= {'SVM__C':[100], 'SVM__gamma':['auto'], 'SVM__kernel': ['rbf'],
                                           'SVM__epsilon':[0.001]}, cv=5)
grid.fit(X_train3, y_train3)
svr_score = grid.score(X_train3,y_train3)
svr_score1 = grid.score(X_test3,y_test3)
y_predicted3 = grid.predict(X_test3)

In [ ]:
print('SVR Model| R2 sq on train set: %.4f'% svr_score)
print('SVR Model| R2 sq on test set: %.4f'% svr_score1)
print('SVR Model| MSE on test set: %.4f'% mean_squared_error(y_test3, y_predicted3))
print('SVR Model| MAE on test set: %.4f'% mean_absolute_error(y_test3, y_predicted3))